# RLO ablation

In [ ]:
import os, gc, time, math, random, json
from pathlib import Path
from itertools import product
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Optimizer
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18
from tqdm.auto import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"vRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
CONFIG = {
    'seed': 42,
    'ablation_epochs': 50,
    'batch_size': 128,
    'data_dir': './data',
    'output_dir': './results_ablation_corrected',
    'checkpoint_dir': './checkpoints_ablation',
}

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG['seed'])
DEVICE = torch.device('cuda')
Path(CONFIG['output_dir']).mkdir(exist_ok=True)
Path(CONFIG['checkpoint_dir']).mkdir(exist_ok=True)

## 1. dataloaders and models

In [ ]:
def get_cifar10_loaders(batch_size=128):
    transform_train = T.Compose([
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ])
    transform_test = T.Compose([
        T.ToTensor(),
        T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ])
    
    trainset = torchvision.datasets.CIFAR10(root=CONFIG['data_dir'], train=True, download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR10(root=CONFIG['data_dir'], train=False, download=True, transform=transform_test)
    
    train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
    test_loader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    
    print(f"Train: {len(trainset)} samples, {len(train_loader)} batches")
    print(f"Test: {len(testset)} samples")
    return train_loader, test_loader

def make_resnet18_cifar10():
    """CIFAR-10 专用 ResNet-18"""
    model = resnet18(num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model

train_loader, test_loader = get_cifar10_loaders(CONFIG['batch_size'])

## 2. RLO-lifted

In [ ]:
def _global_norm_sq(tensors):
    return sum((t * t).sum().item() for t in tensors)


class SmoothLiftedRLO_Diagnosable(Optimizer):
    def __init__(self, params, lr=1e-4, beta1=0.9, beta2=0.99, eta=0.3,
                 weight_decay=0.0, lambda_b=0.1, eps=1e-8, gamma=5.0,
                 use_sign=False,
                 use_global_norm=True,          
                 normalize_belief_properly=False, 
                 collect_diagnostics=False):
        
        defaults = dict(lr=lr, beta1=beta1, beta2=beta2, eta=eta,
                       weight_decay=weight_decay, lambda_b=lambda_b, 
                       eps=eps, gamma=gamma, use_sign=use_sign,
                       use_global_norm=use_global_norm,
                       normalize_belief_properly=normalize_belief_properly)
        super().__init__(params, defaults)
        
        self.total_dim = sum(p.numel() for g in self.param_groups for p in g['params'])
        self.sqrt_dim = math.sqrt(self.total_dim)
        self.collect_diagnostics = collect_diagnostics
        self.diagnostics = defaultdict(list)
    
    def get_diagnostics(self):
        diag = dict(self.diagnostics)
        self.diagnostics = defaultdict(list)
        return diag
    
    @torch.no_grad()
    def step(self, closure=None):
        all_s, all_b, all_d, all_p = [], [], [], []
        all_v_before, all_delta = [], []
        
        for g in self.param_groups:
            for p in g['params']:
                if p.grad is None:
                    continue
                
                state = self.state[p]
                if len(state) == 0:
                    state['m'] = torch.zeros_like(p)  # momentum
                    state['v'] = torch.zeros_like(p)  # lifted velocity
                
                gr = p.grad
                m = state['m']
                c = g['beta1'] * m + (1 - g['beta1']) * gr
                if g['use_sign']:
                    s = c.sign()
                else:
                    s = torch.tanh(g['gamma'] * c)

                delta = gr - m
                delta_norm = delta.norm().clamp(min=g['eps'])
                
                if g['normalize_belief_properly']:
                    b = g['lambda_b'] * (delta / delta_norm) * math.sqrt(p.numel())
                else:
                    b = g['lambda_b'] * (delta / delta_norm)
                
                all_s.append(s)
                all_b.append(b)
                all_p.append((p, g, state))
                all_delta.append(delta)
                
                if self.collect_diagnostics:
                    all_v_before.append(state['v'].clone())
        
        if not all_p:
            return
        
        if self.param_groups[0]['use_global_norm']:
            global_s_norm = math.sqrt(_global_norm_sq(all_s) + 1e-8)
            scale = self.sqrt_dim / global_s_norm
        else:
            scale = 1.0

        if self.collect_diagnostics:
            self.diagnostics['scale'].append(scale)
            self.diagnostics['global_s_norm'].append(math.sqrt(_global_norm_sq(all_s)))

        all_d_tensors = []
        all_v_after = [] 
        
        for idx, ((p, g, state), s, b) in enumerate(zip(all_p, all_s, all_b)):
            # Weight decay
            if g['weight_decay'] != 0:
                p.mul_(1 - g['lr'] * g['weight_decay'])

            d = scale * s + b
            all_d_tensors.append(d)
            state['v'].mul_(1 - g['eta']).add_(d, alpha=g['eta'])
            
            if self.collect_diagnostics:
                all_v_after.append(state['v'].clone())

            p.add_(state['v'], alpha=-g['lr'])
            
            state['m'].mul_(g['beta2']).add_(p.grad, alpha=1 - g['beta2'])
        
        if self.collect_diagnostics:
            # 1. Fiber residual: ||v - d||
            fiber_residual_sq = sum(
                ((v - d) ** 2).sum().item() 
                for v, d in zip(all_v_after, all_d_tensors)
            )
            self.diagnostics['fiber_residual'].append(math.sqrt(fiber_residual_sq))
            
            # 2. Fiber alignment: cos(v, d)
            v_flat = torch.cat([v.flatten() for v in all_v_after])
            d_flat = torch.cat([d.flatten() for d in all_d_tensors])
            cos_sim = F.cosine_similarity(v_flat.unsqueeze(0), d_flat.unsqueeze(0)).item()
            self.diagnostics['fiber_alignment'].append(cos_sim)
            
            d_norm = math.sqrt(sum((d ** 2).sum().item() for d in all_d_tensors))
            self.diagnostics['d_norm'].append(d_norm)

            v_norm = math.sqrt(sum((v ** 2).sum().item() for v in all_v_after))
            self.diagnostics['v_norm'].append(v_norm)
            
            b_norm = math.sqrt(_global_norm_sq(all_b))
            self.diagnostics['belief_norm'].append(b_norm)

## 3. training loops

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return 100.0 * correct / total


def run_experiment(name, optimizer_kwargs, train_loader, test_loader, device, epochs,
                   save_checkpoints=True, collect_diagnostics=True, seed=42):
    set_seed(seed)
    
    print(f"\n{'='*60}")
    print(f"Experiment: {name}")
    print(f"Config: {optimizer_kwargs}")
    print(f"{'='*60}")
    
    model = make_resnet18_cifar10().to(device)
    optimizer = SmoothLiftedRLO_Diagnosable(
        model.parameters(),
        collect_diagnostics=collect_diagnostics,
        **optimizer_kwargs
    )
    criterion = nn.CrossEntropyLoss()
    history = {
        'train_loss': [],
        'train_acc': [],
        'test_acc': [],
        'loss_per_step': [],
        'fiber_residual': [],
        'fiber_alignment': [],
        'd_norm': [],
        'v_norm': [],
        'belief_norm': [],
        'scale': [],
    }
    
    best_acc = 0.0
    best_state = None
    
    pbar = tqdm(range(epochs), desc=name)
    for epoch in pbar:
        model.train()
        epoch_loss = 0.0
        correct, total = 0, 0
        epoch_losses = []
        
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            
            loss_val = loss.item()
            epoch_loss += loss_val
            epoch_losses.append(loss_val)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)

        if collect_diagnostics:
            diag = optimizer.get_diagnostics()
            for key in ['fiber_residual', 'fiber_alignment', 'd_norm', 'v_norm', 'belief_norm', 'scale']:
                if key in diag and len(diag[key]) > 0:
                    history[key].append(np.mean(diag[key]))
                else:
                    history[key].append(0.0)
        
        train_acc = 100.0 * correct / total
        test_acc = evaluate(model, test_loader, device)
        avg_loss = epoch_loss / len(train_loader)
        
        history['train_loss'].append(avg_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        history['loss_per_step'].extend(epoch_losses)
        
        if test_acc > best_acc:
            best_acc = test_acc
            best_state = {
                'epoch': epoch,
                'model_state_dict': {k: v.cpu().clone() for k, v in model.state_dict().items()},
                'test_acc': test_acc,
            }
        
        pbar.set_postfix(
            loss=f"{avg_loss:.3f}",
            train=f"{train_acc:.1f}%",
            test=f"{test_acc:.1f}%",
            best=f"{best_acc:.1f}%"
        )
        
        if save_checkpoints and (epoch + 1) % 10 == 0:
            ckpt_path = f"{CONFIG['checkpoint_dir']}/{name.replace(' ', '_').replace('/', '_')}_epoch{epoch+1}.pt"
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'history': history,
            }, ckpt_path)

    if save_checkpoints and best_state is not None:
        best_path = f"{CONFIG['checkpoint_dir']}/{name.replace(' ', '_').replace('/', '_')}_best.pt"
        torch.save(best_state, best_path)
        print(f"Best model saved: {best_path}")
    
    losses = np.array(history['loss_per_step'])
    stability_metrics = {
        'loss_max': float(np.max(losses)),
        'loss_std': float(np.std(losses)),
        'loss_spike_count': int(np.sum(losses > np.median(losses) * 3)),
        'loss_spike_ratio': float(np.mean(losses > np.median(losses) * 3)),
        'final_acc': history['test_acc'][-1],
        'best_acc': best_acc,
        'acc_std_last10': float(np.std(history['test_acc'][-10:])) if len(history['test_acc']) >= 10 else 0,
    }
    
    print(f"\n✓ {name}: Best={best_acc:.2f}%, Final={history['test_acc'][-1]:.2f}%")
    print(f"  Stability: spike_count={stability_metrics['loss_spike_count']}, loss_std={stability_metrics['loss_std']:.4f}")
    
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()
    
    return {
        'name': name,
        'config': optimizer_kwargs,
        'history': history,
        'stability': stability_metrics,
        'best_acc': best_acc,
        'final_acc': history['test_acc'][-1],
    }

## 4. ablation design

In [ ]:
BASE_CONFIG = {
    'lr': 1e-4,
    'weight_decay': 0.1,
    'beta1': 0.9,
    'beta2': 0.99,
    'gamma': 5.0,
    'lambda_b': 0.2,
    'eta': 0.7, 
    'use_global_norm': True,
    'normalize_belief_properly': False,
    'use_sign': False,
}

ABLATION_MATRIX = {
    # Row 1: Full method (eta=0.3)
    'Full + GlobalNorm': {},
    'Full + NoGlobalNorm': {'lr': 3e-3, 'use_global_norm': False},
    
    # Row 2: No lifted (eta=1)
    'Nolifted(η=1) + GlobalNorm': {'eta': 1.0},
    'Nolifted(η=1) + NoGlobalNorm': {'lr': 3e-3, 'eta': 1.0, 'use_global_norm': False},
    
    # Belief correction
    'NoBelief(λ=0) + GlobalNorm': {'lambda_b': 0.0},
    'NoBelief(λ=0) + NoGlobalNorm': {'lr': 3e-3, 'lambda_b': 0.0, 'use_global_norm': False},
    
    # Properly normalized belief
    'ProperBelief + GlobalNorm': {'normalize_belief_properly': True},
    'ProperBelief + NoGlobalNorm': {'lr': 3e-3, 'normalize_belief_properly': True, 'use_global_norm': False},
    
    # Sign vs Tanh
    'Sign + GlobalNorm': {'use_sign': True},
    'Sign + NoGlobalNorm': {'lr': 3e-3, 'use_sign': True, 'use_global_norm': False},
}

print(f"Total experiments: {len(ABLATION_MATRIX)}")
for name, override in ABLATION_MATRIX.items():
    cfg = {**BASE_CONFIG, **override}
    key_params = f"η={cfg['eta']}, λ={cfg['lambda_b']}, GN={cfg['use_global_norm']}"
    print(f"  {name}: {key_params}")

## 5. experiments and results

In [ ]:
results = {}

for name, override in ABLATION_MATRIX.items():
    cfg = {**BASE_CONFIG, **override}
    
    result = run_experiment(
        name=name,
        optimizer_kwargs=cfg,
        train_loader=train_loader,
        test_loader=test_loader,
        device=DEVICE,
        epochs=CONFIG['ablation_epochs'],
        save_checkpoints=True,
        collect_diagnostics=True,
        seed=CONFIG['seed'],
    )
    results[name] = result


with open(f"{CONFIG['output_dir']}/ablation_results.json", 'w') as f:
    serializable = {}
    for name, r in results.items():
        serializable[name] = {
            'config': r['config'],
            'stability': r['stability'],
            'best_acc': r['best_acc'],
            'final_acc': r['final_acc'],
            'history': {k: v if not isinstance(v, np.ndarray) else v.tolist() 
                       for k, v in r['history'].items()}
        }
    json.dump(serializable, f, indent=2)

print(f"\nResults saved to {CONFIG['output_dir']}/ablation_results.json")

## 6. summary

In [ ]:

core_experiments = [
    'Full + GlobalNorm',
    'Full + NoGlobalNorm', 
    'Nolifted(η=1) + GlobalNorm',
    'Nolifted(η=1) + NoGlobalNorm',
]

print(f"{'Experiment':<35} {'Best Acc':>10} {'Final Acc':>10} {'Spikes':>8} {'Loss Std':>10}")
print("-" * 80)
for name in core_experiments:
    if name in results:
        r = results[name]
        print(f"{name:<35} {r['best_acc']:>10.2f} {r['final_acc']:>10.2f} "
              f"{r['stability']['loss_spike_count']:>8} {r['stability']['loss_std']:>10.4f}")

if all(n in results for n in core_experiments):
    # GlobalNorm 
    diff_gn = results['Full + GlobalNorm']['best_acc'] - results['Nolifted(η=1) + GlobalNorm']['best_acc']
    # NoGlobalNorm
    diff_nogn = results['Full + NoGlobalNorm']['best_acc'] - results['Nolifted(η=1) + NoGlobalNorm']['best_acc']
    
    print(f"  Δ(Full vs Nolifted) with GlobalNorm:    {diff_gn:+.2f}%")
    print(f"  Δ(Full vs Nolifted) without GlobalNorm: {diff_nogn:+.2f}%")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

colors = {
    'Full + GlobalNorm': 'blue',
    'Full + NoGlobalNorm': 'royalblue',
    'Nolifted(η=1) + GlobalNorm': 'red',
    'Nolifted(η=1) + NoGlobalNorm': 'darkred',
}
linestyles = {
    'Full + GlobalNorm': '-',
    'Full + NoGlobalNorm': '--',
    'Nolifted(η=1) + GlobalNorm': '-',
    'Nolifted(η=1) + NoGlobalNorm': '--',
}

# 1. Test Accuracy
ax = axes[0, 0]
for name in core_experiments:
    if name in results:
        ax.plot(results[name]['history']['test_acc'], 
                label=f"{name}: {results[name]['best_acc']:.1f}%",
                color=colors[name], linestyle=linestyles[name], linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Test Accuracy Comparison')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 2. Training Loss
ax = axes[0, 1]
for name in core_experiments:
    if name in results:
        ax.plot(results[name]['history']['train_loss'],
                label=name, color=colors[name], linestyle=linestyles[name], linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title('Training Loss')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 3. Fiber Residual ||v - d||
ax = axes[0, 2]
for name in core_experiments:
    if name in results and 'fiber_residual' in results[name]['history']:
        ax.plot(results[name]['history']['fiber_residual'],
                label=name, color=colors[name], linestyle=linestyles[name], linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('||v - d||')
ax.set_title('Fiber Residual (NAIM Diagnostic)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 4. Fiber Alignment cos(v, d)
ax = axes[1, 0]
for name in core_experiments:
    if name in results and 'fiber_alignment' in results[name]['history']:
        ax.plot(results[name]['history']['fiber_alignment'],
                label=name, color=colors[name], linestyle=linestyles[name], linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('cos(v, d)')
ax.set_title('Fiber Alignment (NAIM Diagnostic)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 5. Scale factor
ax = axes[1, 1]
for name in core_experiments:
    if name in results and 'scale' in results[name]['history']:
        ax.plot(results[name]['history']['scale'],
                label=name, color=colors[name], linestyle=linestyles[name], linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Scale Factor')
ax.set_title('Global Normalization Scale')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 6. Loss Spike
ax = axes[1, 2]
spike_counts = [results[n]['stability']['loss_spike_count'] for n in core_experiments if n in results]
bars = ax.bar(range(len(core_experiments)), spike_counts, 
              color=[colors[n] for n in core_experiments])
ax.set_xticks(range(len(core_experiments)))
ax.set_xticklabels([n.replace(' + ', '\n') for n in core_experiments], fontsize=8)
ax.set_ylabel('Loss Spike Count')
ax.set_title('Training Stability (fewer = better)')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/core_2x2_comparison.png", dpi=150, bbox_inches='tight')
plt.savefig(f"{CONFIG['output_dir']}/core_2x2_comparison.pdf", bbox_inches='tight')
plt.show()

## 7. summary

In [ ]:
summary_data = []
for name, r in results.items():
    cfg = r['config']
    summary_data.append({
        'Experiment': name,
        'η': cfg['eta'],
        'λ_b': cfg['lambda_b'],
        'GlobalNorm': cfg['use_global_norm'],
        'Best Acc (%)': r['best_acc'],
        'Final Acc (%)': r['final_acc'],
        'Acc Std (last 10)': r['stability']['acc_std_last10'],
        'Loss Spikes': r['stability']['loss_spike_count'],
        'Loss Std': r['stability']['loss_std'],
    })

df = pd.DataFrame(summary_data)
df = df.sort_values('Best Acc (%)', ascending=False)

print(df.to_string(index=False))

# 保存为 CSV
df.to_csv(f"{CONFIG['output_dir']}/ablation_summary.csv", index=False)
print(f"\nSaved to {CONFIG['output_dir']}/ablation_summary.csv")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

gn_true = [n for n in results if results[n]['config']['use_global_norm']]
gn_false = [n for n in results if not results[n]['config']['use_global_norm']]

# 1. GlobalNorm=True
ax = axes[0, 0]
for name in gn_true:
    ax.plot(results[name]['history']['test_acc'], label=f"{name}: {results[name]['best_acc']:.1f}%", linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('With Global Normalization')
ax.legend(fontsize=7, loc='lower right')
ax.grid(True, alpha=0.3)

# 2. GlobalNorm=False
ax = axes[0, 1]
for name in gn_false:
    ax.plot(results[name]['history']['test_acc'], label=f"{name}: {results[name]['best_acc']:.1f}%", linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Without Global Normalization')
ax.legend(fontsize=7, loc='lower right')
ax.grid(True, alpha=0.3)

# 3. Best Acc
ax = axes[1, 0]
x = np.arange(len(df))
colors_bar = ['blue' if gn else 'orange' for gn in df['GlobalNorm']]
ax.barh(x, df['Best Acc (%)'], color=colors_bar, alpha=0.7)
ax.set_yticks(x)
ax.set_yticklabels(df['Experiment'], fontsize=7)
ax.set_xlabel('Best Accuracy (%)')
ax.set_title('Best Accuracy Comparison\n(Blue=GlobalNorm, Orange=NoGlobalNorm)')
ax.grid(True, alpha=0.3, axis='x')

# 4. Stability
ax = axes[1, 1]
ax.barh(x, df['Loss Spikes'], color=colors_bar, alpha=0.7)
ax.set_yticks(x)
ax.set_yticklabels(df['Experiment'], fontsize=7)
ax.set_xlabel('Loss Spike Count')
ax.set_title('Training Stability (fewer = better)')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/all_experiments_comparison.png", dpi=150, bbox_inches='tight')
plt.savefig(f"{CONFIG['output_dir']}/all_experiments_comparison.pdf", bbox_inches='tight')
plt.show()

## 8. NAIM 

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

key_experiments = [
    'Full + GlobalNorm',
    'Full + NoGlobalNorm',
    'Nolifted(η=1) + GlobalNorm',
    'Nolifted(η=1) + NoGlobalNorm',
    'StrongFiber(η=1) + GlobalNorm',
    'StrongFiber(η=1) + NoGlobalNorm',
]

diagnostics_to_plot = [
    ('fiber_residual', '||v - d|| (Fiber Residual)', 'log'),
    ('fiber_alignment', 'cos(v, d) (Fiber Alignment)', 'linear'),
    ('d_norm', '||d|| (Target Direction Norm)', 'log'),
    ('v_norm', '||v|| (Velocity Norm)', 'log'),
    ('belief_norm', '||b|| (Belief Term Norm)', 'log'),
    ('scale', 'Global Scale Factor', 'linear'),
]

for idx, (key, title, scale) in enumerate(diagnostics_to_plot):
    ax = axes[idx // 3, idx % 3]
    for name in key_experiments:
        if name in results and key in results[name]['history']:
            data = results[name]['history'][key]
            if len(data) > 0 and any(d != 0 for d in data):
                ax.plot(data, label=name, linewidth=1.5)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(key)
    ax.set_title(title)
    if scale == 'log':
        ax.set_yscale('log')
    ax.legend(fontsize=6)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/naim_diagnostics.png", dpi=150, bbox_inches='tight')
plt.savefig(f"{CONFIG['output_dir']}/naim_diagnostics.pdf", bbox_inches='tight')
plt.show()

## 10.learning rate

In [ ]:
RUN_LR_SWEEP = True

if RUN_LR_SWEEP:
    lr_values = [3e-5, 1e-4, 3e-4, 1e-3, 3e-3]
    variants = [
        ('Full + GN', {'eta': 0.7, 'use_global_norm': True}),
        ('Full + NoGN', {'eta': 0.7, 'use_global_norm': False}),
        ('Nolifted + GN', {'eta': 1.0, 'use_global_norm': True}),
        ('Nolifted + NoGN', {'eta': 1.0, 'use_global_norm': False}),
    ]
    
    lr_sweep_results = {}
    
    for lr in lr_values:
        for var_name, var_override in variants:
            cfg = {**BASE_CONFIG, **var_override, 'lr': lr}
            exp_name = f"{var_name}_lr{lr}"
            
            try:
                result = run_experiment(
                    name=exp_name,
                    optimizer_kwargs=cfg,
                    train_loader=train_loader,
                    test_loader=test_loader,
                    device=DEVICE,
                    epochs=30,
                    save_checkpoints=False,
                    collect_diagnostics=False,
                    seed=CONFIG['seed'],
                )
                lr_sweep_results[exp_name] = {
                    'lr': lr,
                    'variant': var_name,
                    'best_acc': result['best_acc'],
                    'converged': result['best_acc'] > 50,
                }
            except Exception as e:
                print(f"  {exp_name} FAILED: {e}")
                lr_sweep_results[exp_name] = {
                    'lr': lr,
                    'variant': var_name,
                    'best_acc': 0,
                    'converged': False,
                }

    fig, ax = plt.subplots(figsize=(10, 6))
    
    for var_name, _ in variants:
        lrs = []
        accs = []
        for exp_name, r in lr_sweep_results.items():
            if r['variant'] == var_name:
                lrs.append(r['lr'])
                accs.append(r['best_acc'])
        sorted_pairs = sorted(zip(lrs, accs))
        lrs, accs = zip(*sorted_pairs)
        ax.plot(lrs, accs, 'o-', label=var_name, linewidth=2, markersize=8)
    
    ax.set_xscale('log')
    ax.set_xlabel('Learning Rate')
    ax.set_ylabel('Best Accuracy (%)')
    ax.set_title('Stability Region: Accuracy vs Learning Rate')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.axhline(y=50, color='gray', linestyle='--', label='Convergence threshold')
    
    plt.savefig(f"{CONFIG['output_dir']}/lr_sweep_stability.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n stable interval:")
    for var_name, _ in variants:
        max_stable_lr = max(
            [r['lr'] for r in lr_sweep_results.values() 
             if r['variant'] == var_name and r['converged']],
            default=0
        )
        print(f"  {var_name}: max stable learning rate = {max_stable_lr}")

In [ ]:
# Batch Size Search
RUN_BS_SWEEP = True

if RUN_BS_SWEEP:
    bs_values = [32, 64, 128, 256, 512]
    variants = [
        ('Full + GN', {'eta': 0.7, 'use_global_norm': True}),
        ('Full + NoGN', {'eta': 0.7, 'use_global_norm': False, 'lr': 3e-3}),
        ('Nolifted + GN', {'eta': 1.0, 'use_global_norm': True}),
        ('Nolifted + NoGN', {'eta': 1.0, 'use_global_norm': False, 'lr': 3e-3}),
    ]
    
    bs_sweep_results = {}
    
    for bs in bs_values:
        train_loader_bs, test_loader_bs = get_cifar10_loaders(batch_size=bs)
        
        for var_name, var_override in variants:
            cfg = {**BASE_CONFIG, **var_override}
            exp_name = f"{var_name}_bs{bs}"
            
            try:
                result = run_experiment(
                    name=exp_name,
                    optimizer_kwargs=cfg,
                    train_loader=train_loader_bs,
                    test_loader=test_loader_bs,
                    device=DEVICE,
                    epochs=30,
                    save_checkpoints=False,
                    collect_diagnostics=False,
                    seed=CONFIG['seed'],
                )
                bs_sweep_results[exp_name] = {
                    'bs': bs,
                    'variant': var_name,
                    'best_acc': result['best_acc'],
                    'converged': result['best_acc'] > 50,
                }
            except Exception as e:
                print(f"  {exp_name} FAILED: {e}")
                bs_sweep_results[exp_name] = {
                    'bs': bs,
                    'variant': var_name,
                    'best_acc': 0,
                    'converged': False,
                }
                
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for var_name, _ in variants:
        bss = []
        accs = []
        for exp_name, r in bs_sweep_results.items():
            if r['variant'] == var_name:
                bss.append(r['bs'])
                accs.append(r['best_acc'])
        sorted_pairs = sorted(zip(bss, accs))
        bss, accs = zip(*sorted_pairs)
        ax.plot(bss, accs, 'o-', label=var_name, linewidth=2, markersize=8)
    
    ax.set_xscale('log', base=2)
    ax.set_xlabel('Batch Size')
    ax.set_ylabel('Best Accuracy (%)')
    ax.set_title('Accuracy vs Batch Size')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(bs_values)
    ax.set_xticklabels(bs_values)
    
    plt.savefig(f"{CONFIG['output_dir']}/bs_sweep.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nBatch Size Analysis:")
    for var_name, _ in variants:
        best_bs = max(
            [(r['bs'], r['best_acc']) for r in bs_sweep_results.values() if r['variant'] == var_name],
            key=lambda x: x[1],
            default=(0, 0)
        )
        print(f"  {var_name}: best batch size = {best_bs[0]}, acc = {best_bs[1]:.2f}%")

In [ ]:
# LR × Batch Size 2D Grid Search
RUN_GRID_SEARCH = True

if RUN_GRID_SEARCH:
    lr_values_grid = [3e-5, 1e-4, 3e-4, 1e-3, 3e-3]
    bs_values_grid = [32, 64, 128, 256, 512]
    
    variants_grid = [
        ('Full + GN', {'eta': 0.7, 'use_global_norm': True}),
        ('Full + NoGN', {'eta': 0.7, 'use_global_norm': False}),
        ('Nolifted + GN', {'eta': 1.0, 'use_global_norm': True}),
        ('Nolifted + NoGN', {'eta': 1.0, 'use_global_norm': False}),
    ]
    
    grid_results = {var_name: {} for var_name, _ in variants_grid}
    
    for bs in bs_values_grid:
        train_loader_grid, test_loader_grid = get_cifar10_loaders(batch_size=bs)
        
        for lr in lr_values_grid:
            for var_name, var_override in variants_grid:
                cfg = {**BASE_CONFIG, **var_override, 'lr': lr}
                exp_name = f"{var_name}_lr{lr}_bs{bs}"
                
                try:
                    result = run_experiment(
                        name=exp_name,
                        optimizer_kwargs=cfg,
                        train_loader=train_loader_grid,
                        test_loader=test_loader_grid,
                        device=DEVICE,
                        epochs=30,
                        save_checkpoints=False,
                        collect_diagnostics=False,
                        seed=CONFIG['seed'],
                    )
                    grid_results[var_name][(lr, bs)] = result['best_acc']
                except Exception as e:
                    print(f"  {exp_name} FAILED: {e}")
                    grid_results[var_name][(lr, bs)] = 0.0
    
    with open(f"{CONFIG['output_dir']}/grid_search_results.json", 'w') as f:
        json_results = {
            var_name: {f"lr{k[0]}_bs{k[1]}": v for k, v in var_dict.items()}
            for var_name, var_dict in grid_results.items()
        }
        json.dump(json_results, f, indent=2)
    

In [ ]:
def plot_grid_heatmap(grid_results, lr_values, bs_values, output_dir):
    n_variants = len(grid_results)
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()
    all_accs = []
    for var_dict in grid_results.values():
        all_accs.extend(var_dict.values())
    vmin = min(all_accs) if all_accs else 0
    vmax = max(all_accs) if all_accs else 100
    
    for idx, (var_name, var_dict) in enumerate(grid_results.items()):
        ax = axes[idx]

        acc_matrix = np.zeros((len(bs_values), len(lr_values)))
        for i, bs in enumerate(bs_values):
            for j, lr in enumerate(lr_values):
                acc_matrix[i, j] = var_dict.get((lr, bs), 0)

        im = ax.imshow(acc_matrix, cmap='RdYlGn', aspect='auto', vmin=vmin, vmax=vmax)

        ax.set_xticks(range(len(lr_values)))
        ax.set_xticklabels([f'{lr:.0e}' for lr in lr_values], rotation=45, ha='right')
        ax.set_yticks(range(len(bs_values)))
        ax.set_yticklabels(bs_values)
        ax.set_xlabel('Learning Rate')
        ax.set_ylabel('Batch Size')
        ax.set_title(var_name)

        for i in range(len(bs_values)):
            for j in range(len(lr_values)):
                val = acc_matrix[i, j]
                color = 'white' if val < (vmin + vmax) / 2 else 'black'
                ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                       color=color, fontsize=9, fontweight='bold')
    
    fig.suptitle('LR × Batch Size Grid Search Results', fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 0.92, 0.96])
    
    cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
    fig.colorbar(im, cax=cbar_ax, label='Best Accuracy (%)')
    
    plt.savefig(f"{output_dir}/lr_bs_grid_heatmap.png", dpi=150, bbox_inches='tight')
    plt.show()
    return fig

if RUN_GRID_SEARCH:
    plot_grid_heatmap(grid_results, lr_values_grid, bs_values_grid, CONFIG['output_dir'])

In [ ]:

if RUN_GRID_SEARCH:
    print("\n" + "="*60)
    print("LR × Batch Size Grid Search Summary")
    print("="*60)
    
    summary_data = []
    for var_name, var_dict in grid_results.items():
        if var_dict:
            best_key = max(var_dict.keys(), key=lambda k: var_dict[k])
            best_acc = var_dict[best_key]
            summary_data.append({
                'Variant': var_name,
                'Best LR': best_key[0],
                'Best BS': best_key[1],
                'Best Acc (%)': f'{best_acc:.2f}'
            })
            print(f"\n{var_name}:")
            print(f"  最优 LR = {best_key[0]:.0e}, BS = {best_key[1]}")
            print(f"  Best Accuracy = {best_acc:.2f}%")
    
    summary_df = pd.DataFrame(summary_data)
    print("\nSummary Table:")
    print(summary_df.to_string(index=False))
    
    summary_df.to_csv(f"{CONFIG['output_dir']}/grid_search_summary.csv", index=False)

In [ ]:
# smooth lifted rLO eta ablation

etas = [0.1, 0.3, 0.5, 0.7, 1.0]

eta_results = {}

BASE_CONFIG = {
    'lr': 3e-5,
    'weight_decay': 0.1,
    'beta1': 0.9,
    'beta2': 0.99,
    'gamma': 5.0,
    'lambda_b': 0.2,
    'eta': 0.7,
    'use_global_norm': True,
    'normalize_belief_properly': False,
    'use_sign': False,
}

for eta in etas:
    cfg = {**BASE_CONFIG, 'eta': eta}
    name = f"SmoothLifted(η={eta}) + GN"
    
    result = run_experiment(
        name=name,
        optimizer_kwargs=cfg,
        train_loader=train_loader,
        test_loader=test_loader,
        device=DEVICE,
        epochs=CONFIG['ablation_epochs'],
        save_checkpoints=True,
        collect_diagnostics=True,
        seed=CONFIG['seed'],
    )
    eta_results[name] = result
    
    eta_results[name] = {
        'config': result['config'],
        'stability': result['stability'],
        'best_acc': result['best_acc'],
        'final_acc': result['final_acc'],
        'history': {k: v if not isinstance(v, np.ndarray) else v.tolist() 
                   for k, v in result['history'].items()}
    }
    
with open(f"{CONFIG['output_dir']}/eta_ablation_results.json", 'w') as f:
    json.dump(eta_results, f, indent=2)
    

In [ ]:
# plot eta ablation results
fig, ax = plt.subplots(figsize=(8, 6))
for name, r in eta_results.items():
    ax.plot(r['history']['test_acc'], label=f"{name}: {r['best_acc']:.1f}%", linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('η Ablation Study')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.savefig(f"{CONFIG['output_dir']}/eta_ablation.png", dpi=150, bbox_inches='tight')
plt.show()